In [2]:
# =========================================================
# 전이학습 기반 이미지 분류 모델 구현
# ResNet18 Transfer Learning
# =========================================================

# =========================================================
# 1. Google Drive 연결
# =========================================================

from google.colab import drive
drive.mount('/content/drive')


# =========================================================
# 2. 라이브러리 불러오기
# =========================================================

import os
import pandas as pd
import numpy as np

from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from tqdm import tqdm


# =========================================================
# 3. 데이터 경로 설정
# =========================================================

DATA_PATH = '/content/drive/MyDrive/dataset'


# =========================================================
# 4. CSV 파일 불러오기
# =========================================================

train_csv = pd.read_csv(f'{DATA_PATH}/train.csv')
test_csv = pd.read_csv(f'{DATA_PATH}/test.csv')
sample_submission = pd.read_csv(f'{DATA_PATH}/sample_submission.csv')

print(train_csv.head())
print(test_csv.head())


# =========================================================
# 5. 이미지 경로 생성
# =========================================================
# file_name 컬럼 사용

train_csv['image_path'] = train_csv['file_name'].apply(
    lambda x: f'{DATA_PATH}/train/{x}'
)

test_csv['image_path'] = test_csv['file_name'].apply(
    lambda x: f'{DATA_PATH}/test/{x}'
)


# =========================================================
# 6. train / validation 분리
# =========================================================

train_df, valid_df = train_test_split(
    train_csv,
    test_size=0.2,
    stratify=train_csv['label'],
    random_state=42
)

print('Train size :', len(train_df))
print('Valid size :', len(valid_df))


# =========================================================
# 7. 이미지 전처리
# =========================================================
# ResNet18 입력 크기 : 224x224
# ImageNet Normalize 사용

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


valid_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# =========================================================
# 8. Custom Dataset 정의
# =========================================================

class CustomDataset(Dataset):

    def __init__(self, df, transform=None, is_test=False):

        self.df = df
        self.transform = transform
        self.is_test = is_test

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.df.iloc[idx]['image_path']

        image = Image.open(image_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image

        label = self.df.iloc[idx]['label']

        return image, label


# =========================================================
# 9. Dataset / DataLoader 구성
# =========================================================

train_dataset = CustomDataset(
    train_df,
    transform=train_transform
)

valid_dataset = CustomDataset(
    valid_df,
    transform=valid_transform
)

test_dataset = CustomDataset(
    test_csv,
    transform=valid_transform,
    is_test=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# =========================================================
# 10. Device 설정
# =========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)


# =========================================================
# 11. torchvision 모델 선택
# =========================================================
# ResNet18 pretrained 모델 사용

model = models.resnet18(pretrained=True)


# =========================================================
# 12. Feature Extraction
# =========================================================
# 기존 가중치는 고정하고
# 마지막 classifier layer만 학습

for param in model.parameters():
    param.requires_grad = False


# =========================================================
# 13. 마지막 classifier layer 수정
# =========================================================

num_classes = train_csv['label'].nunique()

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)


# =========================================================
# 14. Loss / Optimizer 설정
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)


# =========================================================
# 15. 학습 함수
# =========================================================

def train(model, loader):

    model.train()

    total_loss = 0

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# =========================================================
# 16. Validation 함수
# =========================================================

def validate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    return accuracy


# =========================================================
# 17. 학습 및 검증 수행
# =========================================================

EPOCHS = 3

best_acc = 0

for epoch in range(EPOCHS):

    train_loss = train(model, train_loader)

    val_acc = validate(model, valid_loader)

    print(f'\nEpoch {epoch+1}')
    print(f'Train Loss : {train_loss:.4f}')
    print(f'Validation Accuracy : {val_acc:.4f}')

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            'best_transfer_model.pth'
        )

print('\nBest Validation Accuracy :', best_acc)


# =========================================================
# 18. Best 모델 불러오기
# =========================================================

model.load_state_dict(
    torch.load('best_transfer_model.pth')
)


# =========================================================
# 19. Test 이미지 예측
# =========================================================

model.eval()

predictions = []

with torch.no_grad():

    for images in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.cpu().numpy())


# =========================================================
# 20. submission_transfer.csv 생성
# =========================================================

sample_submission['label'] = predictions

sample_submission.to_csv(
    'submission_transfer.csv',
    index=False
)

print('\nsubmission_transfer.csv 저장 완료!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  file_name  label
0   001.PNG      9
1   002.PNG      4
2   003.PNG      1
3   004.PNG      1
4   005.PNG      6
  file_name
0   001.PNG
1   002.PNG
2   003.PNG
3   004.PNG
4   005.PNG
Train size : 578
Valid size : 145
cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 10/10 [02:22<00:00, 14.22s/it]



Epoch 1
Train Loss : 2.0947
Validation Accuracy : 0.5793


100%|██████████| 10/10 [00:16<00:00,  1.63s/it]



Epoch 2
Train Loss : 1.5676
Validation Accuracy : 0.8207


100%|██████████| 10/10 [00:15<00:00,  1.58s/it]



Epoch 3
Train Loss : 1.2572
Validation Accuracy : 0.8414

Best Validation Accuracy : 0.8413793103448276

submission_transfer.csv 저장 완료!
